# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [1. 2. 3.] [0 1 1] / делит обычным способом, // делит нацело, так как это массив, то делится каждый элемент
2. [4. 4. 4.] 
axis=1 суммирует по строкам, в каждой строке четыре единицы
3.  
[[0 1 2]
 [1 2 3]
 [2 3 4]] 
 NumPy складывает строку [0, 1, 2] со столбцом из этих же чисел (broadcasting) 
4. 6.0 корни: 1, 2, 3, их сумма 6
5. 7.5 среднее квадратов от 1 до 16

In [1]:
import numpy as np
a = np.array([2, 4, 6]); print(a / 2, a // 4)

[1. 2. 3.] [0 1 1]


In [2]:
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))

[4. 4. 4.]


In [3]:
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))

[[0 1 2]
 [1 2 3]
 [2 3 4]]


In [4]:
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())

6.0


In [5]:
# 5
print((np.arange(1, 5) ** 2).mean())

7.5


In [6]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [7]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [8]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [ ]:
# 1
sep = amount[clean & (month == "2026-09")].sum()
october = amount[clean & (month == "2026-10")].sum()
change = (october - sep) / sep * 100
print("Строк осталось:", clean.sum())
print(f"Сентябрь: {sep:.0f}")
print(f"Октябрь: {october:.0f}")
print(f"Изменение: {change:.2f}%")
#Изменения совпадают после исключения Теста 


Строк осталось: 4739
Сентябрь: 35243410
Октябрь: 34419520
Изменение: -2.34%


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [12]:
# 2
categories = np.unique(cat)
months = np.array(["2026-09", "2026-10"])
table = np.array([
    [amount[clean & (cat == c) & (month == m)].sum() for m in months]
    for c in categories])
print(table)
print("Больше выручки:", categories[table.sum(axis=1).argmax()])
change = (table[:, 1] / table[:, 0] - 1) * 100
i = change.argmin()
print("Сильнее падение:", categories[i], f"{change[i]:.2f}%")

[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Больше выручки: Электроника
Сильнее падение: Одежда -7.21%


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [13]:
# 3
counts = np.array([
    [(clean & (cat == c) & (month == m)).sum() for m in months]
    for c in categories])
avg = table / counts
print(avg)
i = avg[:, 1].argmax()
print("Самый высокий чек в октябре:", categories[i], f"{avg[i, 1]:.2f}")

[[10499.43277311 10768.46638655]
 [ 4020.          3815.75250836]
 [ 6187.08530806  6158.59770115]
 [13520.79646018 13045.59815951]
 [32772.74952919 33600.1980198 ]]
Самый высокий чек в октябре: Электроника 33600.20


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [16]:
# 4
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

table = np.array([
    [amount[clean & (city == c) & (month == m)].sum() for m in months]
    for c in cities])

share = table / table.sum(axis=0)
print("Суммы столбцов:", share.sum(axis=0))
i = np.where(cities == "Шымкент")[0][0]
print("Сентябрь:", round(share[i, 0] * 100, 2))
print("Октябрь:", round(share[i, 1] * 100, 2))
print("Изменение:", round((share[i, 1] - share[i, 0]) * 100, 2))

Суммы столбцов: [1. 1.]
Сентябрь: 12.93
Октябрь: 11.75
Изменение: -1.19


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [17]:
# 5
change = table[:, 1] - table[:, 0]
contribution = change / change.sum() * 100

i = np.where(cities == "Шымкент")[0][0]
print("Вклад Шымкента в падение:", round(contribution[i], 2), "%")

Вклад Шымкента в падение: 62.46 %


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6
share = table / table.sum(axis=1, keepdims=True)
print(share)
print("Суммы строк:", share.sum(axis=1))
"""Не совпадает количество строк"""

[[0.50357019 0.49642981]
 [0.49756469 0.50243531]
 [0.50513538 0.49486462]
 [0.48788975 0.51211025]
 [0.51971739 0.48028261]
 [0.52991168 0.47008832]]
Суммы строк: [1. 1. 1. 1. 1. 1.]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [19]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()
outliers = np.abs(z) > 3
print("Количество:", outliers.sum())
print("Доля:", round(outliers.mean() * 100, 2), "%")
print("Среднее ≈ 0:", np.isclose(z.mean(), 0))
print("std ≈ 1:", np.isclose(z.std(), 1))

Количество: 87
Доля: 1.84 %
Среднее ≈ 0: True
std ≈ 1: True


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8